# Fitting a function from samples — least squares and stochastic gradient descent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/reinforcement_learning/function_approximation_sgd.ipynb)

Every learning method in these notes replaces a table by a **parametric function** $\hat f(x \mid w)$ and adjusts the parameters $w$ from samples. This notebook does that on the smallest possible case, in plain `numpy`, so that every line maps onto the equations: a scalar function $f(x)$ observed through $N$ noisy samples
$$y_i = f(x_i) + \text{noise},$$
approximated by a weighted sum of fixed basis functions,
$$\hat f(x \mid w) = \sum_k w_k\, \phi_k(x) = w^T \phi(x),$$
with the weights chosen to minimize the mean squared prediction error
$$\mathcal{L}(w) = \frac{1}{N} \sum_{i=1}^N \frac{1}{2}\big( y_i - w^T \phi(x_i) \big)^2 .$$

Two ways to minimize $\mathcal{L}$ are compared: **least squares**, which solves for $w$ in one line because $\mathcal{L}$ is quadratic in $w$, and **stochastic gradient descent** (SGD), which takes one small step per sample and is the algorithm that survives when the approximator is not linear in its parameters — a neural network — or when the samples arrive one at a time.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 1. Samples of an unknown function

The function to learn is $f(x) = \sin 3x$ on $[-1, 1]$; the algorithms only ever see the $N$ noisy samples.

In [ ]:
N = 200  # number of samples
SIGMA = 0.2  # noise level
rng = np.random.default_rng(0)


def f(x):
    return np.sin(3.0 * x)


X = rng.uniform(-1.0, 1.0, N)
y = f(X) + SIGMA * rng.normal(size=N)

x_plot = np.linspace(-1.0, 1.0, 200)
plt.figure(figsize=(6, 3.6))
plt.plot(X, y, ".", alpha=0.5, label="samples $(x_i, y_i)$")
plt.plot(x_plot, f(x_plot), "k", label="$f(x)$")
plt.xlabel("$x$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. A linear approximation on polynomial bases

The bases are the monomials $\phi(x) = [1,\; x,\; x^2,\; \dots,\; x^d]$; the approximation $\hat f(x \mid w) = w^T \phi(x)$ is a polynomial of degree $d$ whose coefficients are the weights. Stacking $\phi(x_i)^T$ for every sample gives the regression matrix $\Phi$ ($N \times (d+1)$), so that the vector of predictions is $\Phi w$.

In [ ]:
DEGREE = 5


def phi(x):
    """Feature vector of one point: the monomials 1, x, ..., x^d."""
    return np.array([x**k for k in range(DEGREE + 1)])


Phi = np.array([phi(x) for x in X])  # one row phi(x_i)^T per sample


def f_hat(x, w):
    return w @ phi(x)


def loss(w):
    """Mean squared prediction error L(w) over the N samples."""
    return np.mean(0.5 * (y - Phi @ w) ** 2)

## 3. Least squares: the closed-form minimum

$\mathcal{L}$ is quadratic in $w$, so setting its gradient to zero gives the normal equations $\Phi^T \Phi\, w = \Phi^T y$. `np.linalg.lstsq` solves them.

In [ ]:
w_ls = np.linalg.lstsq(Phi, y, rcond=None)[0]

print("least-squares weights:", np.round(w_ls, 3))
print(f"loss: {loss(w_ls):.4f}   (noise floor sigma^2/2 = {SIGMA**2 / 2:.4f})")

plt.figure(figsize=(6, 3.6))
plt.plot(X, y, ".", alpha=0.3, label="samples")
plt.plot(x_plot, f(x_plot), "k", label="$f(x)$")
plt.plot(x_plot, [f_hat(x, w_ls) for x in x_plot], "r", label="least-squares fit")
plt.xlabel("$x$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Stochastic gradient descent: one sample at a time

The gradient of the loss of a **single** sample is $-\big(y_i - w^T\phi(x_i)\big)\,\phi(x_i)$. SGD moves the weights against it, with a learning rate $\eta$:
$$w \leftarrow w + \eta\,\big( y_i - w^T \phi(x_i) \big)\, \phi(x_i) .$$
One pass over the shuffled samples is an *epoch*. The learning rate is passed as a function of the update count $k$, so a decreasing schedule is one line away.

In [ ]:
def sgd(eta, n_epochs=20, seed=0):
    """SGD on the loss, one sample per update. Returns the weights and the loss after every update."""
    rng = np.random.default_rng(seed)
    w = np.zeros(DEGREE + 1)
    history = []
    for epoch in range(n_epochs):
        for i in rng.permutation(N):
            k = len(history)
            error = y[i] - w @ Phi[i]  # prediction error on one sample
            w = w + eta(k) * error * Phi[i]  # gradient step
            history.append(loss(w))
    return w, np.array(history)


plt.figure(figsize=(7, 3.8))
for eta_0 in (0.01, 0.05, 0.2):
    w_sgd, history = sgd(eta=lambda k: eta_0)
    plt.semilogy(history, label=f"$\\eta$ = {eta_0}")
    print(f"eta = {eta_0:<5}  loss after {len(history)} updates: {history[-1]:.4f}   weights: {np.round(w_sgd, 2)}")
plt.axhline(loss(w_ls), color="k", linestyle=":", label="least squares")
plt.xlabel("update $k$")
plt.ylabel("$\\mathcal{L}(w)$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

A small $\eta$ converges slowly; a large one converges fast but keeps bouncing around the minimum, because each step follows the gradient of one noisy sample rather than of the mean loss. Least squares reaches the exact minimum in one line — but only because the model is linear in $w$.

## 5. Things to try

1. **Variable learning rate.** Replace the constant by a decreasing schedule, $\eta_k = \eta_0 / (1 + k/\tau)$, and pick $\eta_0$ and $\tau$ so that SGD both starts fast and settles at the least-squares loss. What does the schedule need to satisfy for the bouncing to die out?
2. **Number of samples.** Rerun with `N = 20`, then `N = 2000`. Compare the least-squares fit with $f$ on the plot, and the final loss with the noise floor $\sigma^2/2$. Which quantity depends on $N$, and which one does not?
3. **Noise level.** Rerun with `SIGMA = 0.05` and `SIGMA = 0.5`. How does the noise change the loss reached by SGD, its bouncing, and the learning rate you can afford?
4. **Degree.** Try `DEGREE = 2` and `DEGREE = 12`. One under-fits, the other starts to follow the noise; see where each fails on the plot, then check how the higher degree changes the conditioning of $\Phi$ and the largest usable $\eta$.